# DAR + STSG: train lại SFT trên Kaggle

**Adapt vào dữ liệu và nhiệm vụ phụ SFT.** Mỗi video có target DAR gốc và
target STSG/caption. Mọi nhánh bắt đầu độc lập từ **Qwen2.5-VL-3B-Instruct**.
Không cần checkpoint DAR-SFT/GRPO cũ. Inference dùng một lượt video → DAR JSON;
graph không được đưa vào input benchmark. GRPO không nằm trong pilot này.

Mặc định chỉ chạy `ARM='stsg'`: teacher VideoLLaMA3-7B sinh graph rồi student học SFT.
Đổi `ARM='caption'` hoặc `'stsg_no_links'` để chạy riêng; không chạy lại baseline.
So sánh chính: STSG − caption. Đây là prototype lấy cảm hứng STEP/VoT, không phải
MotionEpic hay toàn bộ VoT. Graph text không có bbox/tracker và chưa được xác minh
thực tế chỉ vì pass schema.

Notebook chứa code cần thiết. Chọn GPU Kaggle, attach base model, train/test JSONL,
video và offline wheelhouse như notebook GRPO hiện tại. `smoke` đã xác minh kỹ thuật;
`full` chạy toàn bộ train.jsonl trừ 64 video dev. Chưa có kết luận cải thiện.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

# Sửa các đường dẫn theo Kaggle Inputs. BASE_MODEL phải là backbone Instruct gốc.
TEACHER_MODEL = Path('/kaggle/input/datasets/vucongaaa/videollama3-7b')
TEACHER_4BIT = False  # The RTX PRO 6000 has sufficient VRAM for the FP16 teacher.
TEACHER_GPUS = '0'  # Set '0,1' with TEACHER_4BIT=False for two GPUs.
TEACHER_WHEELHOUSE = None  # Auto-detect transformers 4.57.1 under /kaggle/input; set a Path to override.
TEACHER_VENV = Path('/tmp/dar_videollama3_venv')
BASE_MODEL = Path('/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2')
TRAIN_JSONL = Path('/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl')
TEST_JSONL = Path('/kaggle/input/datasets/vucongaaa/dar-annotation/test.jsonl')
VIDEO_ROOT = Path('/kaggle/input/datasets/vucongaaa/vce-original-videos/videos')
WHEELHOUSE = Path('/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128')
RUNTIME = Path('/tmp/dar_stsg_python')
PROJECT = Path('/kaggle/working/dar_stsg_project')
MODE = 'full'  # Entire train.jsonl except the frozen 64-video dev split.
assert MODE in ('smoke', 'pilot', 'full')
TRAIN_SIZE, DEV_SIZE = ((8, 4) if MODE == 'smoke' else
                        (256, 64) if MODE == 'pilot' else (-1, 64))
NUM_TRAIN_EPOCHS = 0.5
ARM = 'stsg'  # Choose stsg, caption, or stsg_no_links; never runs baseline automatically.
assert ARM in ('stsg', 'caption', 'stsg_no_links')
ARMS = [ARM]
WORK = Path('/kaggle/working/dar_videollama3_full_epoch05_' + MODE + '_' + ARM)
MODEL_OUTPUT_ROOT = Path('/kaggle/temp/dar_stsg_models_' + MODE + '_' + ARM)
LEARNING_RATE = 1e-5
GRADIENT_ACCUMULATION_STEPS = 32  # Matches dar_kaggle_sft_clean.ipynb.
SEED = 1234
for path in (TEACHER_MODEL, BASE_MODEL, TRAIN_JSONL, TEST_JSONL, VIDEO_ROOT, WHEELHOUSE):
    assert path.exists(), f'Sửa đường dẫn chưa tồn tại: {path}'
assert (TEACHER_MODEL / 'config.json').exists()
assert list(TEACHER_MODEL.glob('modeling*.py')), 'Attach full teacher snapshot with custom code'
assert (BASE_MODEL / 'config.json').exists()
assert list(BASE_MODEL.glob('*.safetensors')), 'Cần full base weights, không phải LoRA adapter'

# The Kaggle model mount may omit preprocessor_config.json. Keep the read-only
# mount intact and make a lightweight local view of its files for ms-swift.
BASE_MODEL_SOURCE = BASE_MODEL
BASE_MODEL = Path('/tmp/dar_stsg_base_model')
BASE_MODEL.mkdir(parents=True, exist_ok=True)
for source in BASE_MODEL_SOURCE.iterdir():
    if source.name != 'preprocessor_config.json':
        target = BASE_MODEL / source.name
        if not target.exists():
            target.symlink_to(source, target_is_directory=source.is_dir())
preprocessor_source = BASE_MODEL_SOURCE / 'preprocessor_config.json'
preprocessor = (json.loads(preprocessor_source.read_text()) if preprocessor_source.exists()
                else dict(min_pixels=3136, max_pixels=12845056, patch_size=14,
                          temporal_patch_size=2, merge_size=2,
                          image_mean=[0.48145466, 0.4578275, 0.40821073],
                          image_std=[0.26862954, 0.26130258, 0.27577711]))
preprocessor['image_processor_type'] = 'Qwen2VLImageProcessor'
preprocessor.setdefault('processor_class', 'Qwen2_5_VLProcessor')
(BASE_MODEL / 'preprocessor_config.json').write_text(json.dumps(preprocessor))
print('Student model view:', BASE_MODEL, 'preprocessor:', preprocessor)


Runtime riêng, dùng wheelhouse có sẵn; không thay NumPy/Torch đang load trong
kernel. Tất cả tác vụ ML chạy ở subprocess. Không thêm `ms-swift/` checkout 4.0
vào PYTHONPATH của runtime 3.12.5.

In [ ]:
"""Student bootstrap cell; embedded verbatim by build_notebook.py."""
import tempfile
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

# Preserve Kaggle's package directories (including sitecustomize dependencies).
# Do not inherit an old student/teacher overlay from a previous bootstrap.
base_paths = [p for p in sys.path if p and Path(p).is_dir()
              and ('site-packages' in p or 'dist-packages' in p)]
inherited_paths = os.environ.get('PYTHONPATH', '').split(os.pathsep)
base_paths = list(dict.fromkeys(
    p for p in [*inherited_paths, *base_paths] if p
    and not Path(p).resolve().is_relative_to(Path('/kaggle/working'))))
base_env = dict(os.environ, PYTHONPATH=os.pathsep.join(base_paths),
                PIP_NO_INDEX='1', PIP_DISABLE_PIP_VERSION_CHECK='1',
                HF_HUB_OFFLINE='1', TOKENIZERS_PARALLELISM='false')

# Check the actual base interpreter, not torch already imported in the kernel.
subprocess.run([sys.executable, '-c',
    "import torch; print('Base torch:', torch.__version__, torch.__file__); "
    "assert torch.version.cuda is not None, 'Kaggle base Torch is CPU-only; use a GPU image with CUDA Torch'; "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing'"],
    check=True, env={**base_env, 'CUDA_VISIBLE_DEVICES': TEACHER_GPUS})

supported_tags = set(sys_tags())
wheels = []
student_pins = {'transformers': '4.57.1', 'ms-swift': '3.12.5'}
for wheel in sorted(WHEELHOUSE.rglob('*.whl')):
    name, version, _, tags = parse_wheel_filename(wheel.name)
    # Reuse Kaggle's matched CUDA stack; never overlay it with a CPU wheel.
    if name in {'torch', 'torchvision', 'torchaudio', 'triton', 'torchao'} or name.startswith('nvidia-'):
        continue
    if not (tags & supported_tags) or (name == 'packaging' and str(version) == '26.3'):
        continue
    # Do not pass two versions of one package to pip if additional wheels are attached.
    if name in student_pins and str(version) != student_pins[name]:
        continue
    wheels.append(wheel)
assert wheels, f'No compatible student wheels in {WHEELHOUSE}'
assert any(parse_wheel_filename(p.name)[0] == 'transformers'
           and str(parse_wheel_filename(p.name)[1]) == student_pins['transformers']
           for p in wheels), 'Student wheelhouse needs transformers 4.57.1'

# A fresh overlay prevents CPU Torch left by an earlier cell from being imported.
RUNTIME.mkdir(parents=True, exist_ok=True)
student_site = Path(tempfile.mkdtemp(prefix='student-', dir=RUNTIME))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                '--only-binary=:all:', '--target', str(student_site), *map(str, wheels)],
               check=True, env=base_env)
env = dict(base_env, PYTHONPATH=os.pathsep.join([str(student_site), *base_paths]),
           CUDA_VISIBLE_DEVICES='0')
def run(*args, extra_env=None):
    subprocess.run(list(map(str, args)), check=True, env={**env, **(extra_env or {})})
run(sys.executable, '-c',
    "import torch, transformers, swift, peft, qwen_vl_utils; "
    "print('Student:', torch.__version__, transformers.__version__, swift.__version__); "
    "print('Torch path:', torch.__file__); print('Transformers path:', transformers.__file__); "
    "assert torch.version.cuda is not None and torch.cuda.is_available(), 'Student requires CUDA Torch'; "
    "assert transformers.__version__ == '4.57.1', transformers.__file__; "
    "assert swift.__version__ == '3.12.5'; print(torch.cuda.get_device_name(0))")


In [ ]:
run(sys.executable, '-c',
    "import sys; from transformers import AutoProcessor; "
    "p = AutoProcessor.from_pretrained(sys.argv[1], local_files_only=True); "
    "print('Student processor preflight:', type(p).__name__, type(p.image_processor).__name__)",
    BASE_MODEL)


Teacher riêng: VideoLLaMA3-7B, Transformers 4.57.1; student dùng 4.57.1.
Notebook chạy **Internet Off**. `TEACHER_WHEELHOUSE=None` tự tìm wheel 4.57.1
trong Kaggle Inputs; dataset runtime riêng cung cấp ffmpeg-python/future.
Nếu thiếu wheel, pip dừng tại bước kiểm tra offline, không thử kết nối PyPI.
Giữ CUDA Torch có sẵn của Kaggle; không cài lại Torch CPU từ wheelhouse.
Mỗi lần chạy bootstrap tạo thư mục runtime mới, giữ nguyên các runtime cũ.
Teacher tạo venv bằng `--without-pip`, dùng pip kernel quản lý qua `--python`
([pip docs](https://pip.pypa.io/en/stable/topics/python-option/), cần pip >=22.3).
Đường dẫn package teacher được ưu tiên trước package Kaggle; không xoá đường dẫn
hỗ trợ của Kaggle. Kiểm tra phiên bản và đường dẫn import trước khi chạy model.
Full teacher snapshot phải có weights, tokenizer, configs và custom Python files.
Teacher FP16 chạy trên GPU đã chọn; NF4 chỉ là tùy chọn khi có wheel tương thích.

In [ ]:
"""Offline teacher bootstrap cell; embedded verbatim by build_notebook.py."""
import tempfile

requirements = ['transformers==4.57.1', 'accelerate==1.11.0',
                'decord==0.6.0', 'numpy>=1.26', 'einops==0.8.1', 'ffmpeg-python==0.2.0']
if TEACHER_WHEELHOUSE is None:
    teacher_candidates = sorted(Path('/kaggle/input').rglob('transformers-4.57.1-*.whl'))
    assert teacher_candidates, (
        'Internet Off: no transformers-4.57.1 wheel was found under /kaggle/input. '
        'Attach a Kaggle Dataset containing the teacher wheelhouse, then rerun this cell.')
    candidate_roots = sorted({p.parent for p in teacher_candidates})
    assert len(candidate_roots) == 1, (
        f'Found transformers 4.57.1 in multiple directories: {candidate_roots}. '
        'Set TEACHER_WHEELHOUSE explicitly in the configuration cell.')
    teacher_wheelhouse = candidate_roots[0]
    print('Auto-detected teacher wheelhouse:', teacher_wheelhouse)
else:
    teacher_wheelhouse = Path(TEACHER_WHEELHOUSE)
    assert teacher_wheelhouse.is_dir(), f'Missing teacher wheelhouse: {teacher_wheelhouse}'
teacher_wheels = sorted(teacher_wheelhouse.rglob('*.whl'))
assert any(p.name.startswith('transformers-4.57.1-') for p in teacher_wheels), (
    f'Missing transformers-4.57.1 wheel in {teacher_wheelhouse}; attach teacher wheels before running')
install_flags = ['--no-index', '--only-binary=:all:']
for directory in sorted({p.parent for p in Path('/kaggle/input').rglob('*.whl')}):
    install_flags.extend(['--find-links', str(directory)])

# Create the interpreter first. No ensurepip, no deletion of earlier environments.
TEACHER_VENV.mkdir(parents=True, exist_ok=True)
teacher_venv = Path(tempfile.mkdtemp(prefix='teacher-', dir=TEACHER_VENV))
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages',
                '--without-pip', str(teacher_venv)], check=True, env=base_env)
teacher_python = teacher_venv / 'bin/python'
assert teacher_python.is_file(), f'Failed to create teacher interpreter: {teacher_python}'
teacher_site = teacher_venv / f'lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages'
teacher_env = dict(base_env,
    PYTHONPATH=os.pathsep.join([str(teacher_site), *base_paths]),
    CUDA_VISIBLE_DEVICES=TEACHER_GPUS)

# Pin the existing CUDA stack so dependency resolution cannot replace it.
stack_probe = subprocess.run([sys.executable, '-c',
    "import importlib.metadata as m, json; "
    "print(json.dumps({d.metadata['Name']: d.version for d in m.distributions() "
    "if d.metadata['Name'] and (d.metadata['Name'].lower() in "
    "('torch', 'torchvision', 'torchaudio', 'triton') or d.metadata['Name'].lower().startswith('nvidia-'))}))"],
    check=True, env=base_env, text=True, capture_output=True)
cuda_stack = json.loads(stack_probe.stdout)
constraints = teacher_venv / 'cuda-constraints.txt'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in cuda_stack.items()))

# Kernel pip >=22.3 can manage a venv that has no pip of its own.
# Resolve offline first; missing wheels fail without partially installing packages.
pip_command = [sys.executable, '-m', 'pip', '--python', str(teacher_python),
               'install', *install_flags, '--constraint', str(constraints), *requirements]
subprocess.run([*pip_command, '--dry-run'], check=True, env=teacher_env)
subprocess.run(pip_command, check=True, env=teacher_env)

subprocess.run([str(teacher_python), '-c',
    "import sys, torch, transformers, decord, cv2; "
    "print('Teacher Python:', sys.executable); "
    "print('Teacher Transformers:', transformers.__version__, transformers.__file__); "
    "print('Teacher Torch:', torch.__version__, torch.__file__); "
    "assert transformers.__version__ == '4.57.1', 'Wrong teacher Transformers: ' + transformers.__file__; "
    "assert torch.version.cuda is not None and torch.cuda.is_available(), 'Teacher requires CUDA Torch'; "
    "print('Teacher GPUs:', torch.cuda.device_count())"], check=True, env=teacher_env)


Cell sau giải nén code đã nhúng vào notebook; không phụ thuộc bản DAR cũ trong
Kaggle Inputs. Các file hiện có khác nội dung sẽ làm cell dừng để tránh trộn version.

In [ ]:
FILES = {'experiments/dar_stsg/core.py': '"""Training-only STSG auxiliary supervision. No torch dependency in data tools."""\nfrom __future__ import annotations\n\nimport ast\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[2]\nARMS = (\'baseline\', \'caption\', \'stsg\', \'stsg_no_links\')\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\n\ndef write_jsonl(path, rows):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as stream:\n        for row in rows:\n            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef official_functions():\n    """Load pure prompt/scoring functions without importing the vLLM entry point."""\n    names = {\'EMOTION_CANDIDATES\', \'build_eval_prompt\', \'calculate_iou\',\n             \'evaluate_single_video\', \'compute_overall_metrics\',\n             \'merge_adjacent_same_emotion_segments\', \'validate_and_fix_segments\'}\n    tree = ast.parse((ROOT / \'test.py\').read_text(encoding=\'utf-8\'))\n    selected = []\n    for node in tree.body:\n        if isinstance(node, ast.FunctionDef) and node.name in names:\n            selected.append(node)\n        elif isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id in names for t in node.targets):\n            selected.append(node)\n    from typing import List, Dict\n    scope = {\'List\': List, \'Dict\': Dict}\n    exec(compile(ast.Module(body=selected, type_ignores=[]), str(ROOT / \'test.py\'), \'exec\'), scope)\n    return scope\n\n\ndef number(x):\n    if isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x):\n        raise ValueError(\'Expected a finite numeric value\')\n    return float(x)\n\n\ndef parse_json(text):\n    text = text.strip()\n    if text.startswith(\'```\') and text.endswith(\'```\'):\n        text = text.split(\'\\n\', 1)[1].rsplit(\'```\', 1)[0].strip()\n    obj = json.loads(text)\n    if not isinstance(obj, dict):\n        raise ValueError(\'Expected a JSON object\')\n    return obj\n\n\ndef validate_graph(graph, duration):\n    """Structural validation only; it does not certify visual truth."""\n    if set(graph) != {\'entities\', \'events\', \'temporal_links\'}:\n        raise ValueError(\'Invalid STSG top-level keys\')\n    entities, events, links = (graph[k] for k in (\'entities\', \'events\', \'temporal_links\'))\n    if not all(isinstance(x, list) for x in (entities, events, links)):\n        raise ValueError(\'Graph fields must be lists\')\n    if not 1 <= len(entities) <= 8 or not 1 <= len(events) <= 8 or len(links) > 16:\n        raise ValueError(\'Graph size outside pilot limits\')\n    entity_ids = set()\n    for entity in entities:\n        if set(entity) != {\'id\', \'label\'} or not all(isinstance(v, str) and v.strip() for v in entity.values()):\n            raise ValueError(\'Invalid entity\')\n        if entity[\'id\'] in entity_ids:\n            raise ValueError(\'Duplicate entity ID\')\n        entity_ids.add(entity[\'id\'])\n    event_ids = {}\n    for event in events:\n        if set(event) != {\'id\', \'start_time\', \'end_time\', \'observation\', \'relations\'}:\n            raise ValueError(\'Invalid event keys\')\n        if not isinstance(event[\'id\'], str) or not event[\'id\'] or event[\'id\'] in event_ids:\n            raise ValueError(\'Invalid/duplicate event ID\')\n        start, end = number(event[\'start_time\']), number(event[\'end_time\'])\n        if not 0 <= start < end <= duration + .051:\n            raise ValueError(\'Event outside video\')\n        if not isinstance(event[\'observation\'], str) or not event[\'observation\'].strip():\n            raise ValueError(\'Missing visual observation\')\n        if not isinstance(event[\'relations\'], list) or len(event[\'relations\']) > 8:\n            raise ValueError(\'Invalid relations\')\n        for edge in event[\'relations\']:\n            if not isinstance(edge, list) or len(edge) != 3 or not all(isinstance(v, str) for v in edge):\n                raise ValueError(\'Expected subject-predicate-object triple\')\n            if edge[0] not in entity_ids or edge[2] not in entity_ids or not edge[1].strip():\n                raise ValueError(\'Dangling relation\')\n        event_ids[event[\'id\']] = event\n    for edge in links:\n        if not isinstance(edge, list) or len(edge) != 3:\n            raise ValueError(\'Invalid temporal edge\')\n        a, kind, b = edge\n        if a not in event_ids or b not in event_ids or a == b or kind != \'before\':\n            raise ValueError(\'Invalid temporal references\')\n        if event_ids[a][\'end_time\'] > event_ids[b][\'start_time\'] + .051:\n            raise ValueError(\'Contradictory before edge\')\n    return graph\n\n\ndef evidence_prompt(duration, kind):\n    common = (f\'Observe this SILENT video of duration {duration:.1f}s. \'\n              \'Describe only visible entities, interactions, appearance, camera/scene changes and events. \'\n              \'Do not use audio, viewer emotion labels, inferred intentions or imagined outcomes. \'\n              \'Omit uncertain details. An event boundary is not necessarily an emotion boundary. \')\n    if kind == \'caption\':\n        return common + \'Return ONLY JSON {"caption":"a concise chronological visual description, at most 250 words"}.\'\n    return common + \'\'\'Return ONLY one compact JSON object, with no markdown or explanation.\nIt must have exactly three top-level keys: entities, events, temporal_links.\nentities is a nonempty array of at most 8 objects, each with exactly id and label.\nUse a distinct short string ID for each visible entity and a specific visual label.\nevents is a nonempty chronological array of at most 8 objects, each with exactly\nid, start_time, end_time, observation, relations. Use distinct event IDs.\nTimes are numeric seconds within the video, with start_time strictly before end_time.\nThe observation must describe a concrete visible state or change.\nrelations is an array of at most 8 triples of strings: subject entity ID,\nvisible relation, object entity ID. Both IDs MUST occur in entities.\ntemporal_links is an array of at most 16 triples of strings: earlier event ID,\nthe word before, later event ID. Both IDs MUST occur in events and the earlier\nevent must end no later than the later event starts.\nUse empty arrays for relations or temporal_links when no justified edge exists.\nNever invent an ID merely to fill a relation or temporal link. Never copy schema\ndescriptions into labels or observations. Do not invent bounding boxes.\'\'\'\n\n\ndef dar_prompt(duration):\n    return official_functions()[\'build_eval_prompt\'](duration)\n\n\ndef annotation(row, video_root=None):\n    video = row.get(\'video\') or row.get(\'video_path\')\n    if not isinstance(video, str):\n        raise ValueError(\'Missing video\')\n    basename = video.replace(\'\\\\\', \'/\').split(\'/\')[-1]\n    path = str(Path(video_root) / basename) if video_root else video\n    duration = number(row.get(\'video_duration\'))  # Never infer duration from GT boundaries.\n    if duration <= 0:\n        raise ValueError(\'Invalid duration\')\n    answers = [x[\'value\'] for x in row.get(\'conversations\', []) if x.get(\'from\') == \'gpt\']\n    target = parse_json(answers[-1]) if answers else {\'segments\': row.get(\'gt_segments\')}\n    if not isinstance(target.get(\'segments\'), list) or not target[\'segments\']:\n        raise ValueError(\'Missing DAR target\')\n    return dict(video_id=Path(basename).stem, video_path=path, video_duration=duration, target=target)\n\n\ndef sft_rows(row, evidence, arm):\n    def sample(prompt, target, suffix):\n        return {\'id\': row[\'video_id\'] + \':\' + suffix,\n                \'messages\': [{\'role\': \'user\', \'content\': \'<video>\\n\' + prompt},\n                             {\'role\': \'assistant\', \'content\': json.dumps(target, ensure_ascii=False)}],\n                \'videos\': [row[\'video_path\']]}\n    answer = sample(dar_prompt(row[\'video_duration\']), row[\'target\'], \'dar\')\n    if arm == \'baseline\':\n        auxiliary = copy.deepcopy(answer)\n        auxiliary[\'id\'] = row[\'video_id\'] + \':dar-repeat\'\n    elif arm == \'caption\':\n        auxiliary = sample(evidence_prompt(row[\'video_duration\'], \'caption\'), evidence[\'caption\'], \'caption\')\n    else:\n        graph = copy.deepcopy(evidence[\'stsg\'])\n        prompt = evidence_prompt(row[\'video_duration\'], \'stsg\')\n        if arm == \'stsg_no_links\':\n            graph[\'temporal_links\'] = []\n            prompt += \'\\nFor this task leave temporal_links empty; retain event timestamps.\'\n        auxiliary = sample(prompt, graph, arm)\n    return [answer, auxiliary]\n', 'experiments/dar_stsg/prepare.py': '"""Freeze pilot split, then create matched SFT arms from train-only evidence."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom core import ARMS, annotation, digest, read_jsonl, sft_rows, validate_graph, write_jsonl\n\n\ndef split(args):\n    rows = [annotation(r, args.video_root) for r in read_jsonl(args.train)]\n    ids = [r[\'video_id\'] for r in rows]\n    if len(ids) != len(set(ids)):\n        raise ValueError(\'Duplicate source video IDs; split at video level first\')\n    heldout = {annotation(r)[\'video_id\'] for r in read_jsonl(args.test)}\n    if set(ids) & heldout:\n        raise ValueError(\'Train/test video overlap\')\n    rows.sort(key=lambda r: digest([args.seed, r[\'video_id\']]))\n    train_size = len(rows) - args.dev_size if args.train_size == -1 else args.train_size\n    if train_size < 1 or args.dev_size < 1 or len(rows) < train_size + args.dev_size:\n        raise ValueError(\'Insufficient rows or invalid requested sizes\')\n    dev, train = rows[:args.dev_size], rows[args.dev_size:args.dev_size + train_size]\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    write_jsonl(out / \'train.jsonl\', train)\n    write_jsonl(out / \'dev.jsonl\', dev)\n    # The teacher receives no labels, GT event boundaries or previous model predictions.\n    write_jsonl(out / \'teacher_inputs.jsonl\', [{k: r[k] for k in (\'video_id\', \'video_path\', \'video_duration\')} for r in train])\n    (out / \'split.json\').write_text(json.dumps(dict(seed=args.seed, train=len(train), dev=len(dev),\n        train_source_sha256=digest(read_jsonl(args.train)), test_source_sha256=digest(read_jsonl(args.test)),\n        train_ids=[r[\'video_id\'] for r in train], dev_ids=[r[\'video_id\'] for r in dev]), indent=2), encoding=\'utf-8\')\n\n\ndef build(args):\n    arms = getattr(args, \'arms\', ARMS)\n    need_graph = any(a in (\'stsg\', \'stsg_no_links\') for a in arms)\n    need_caption = \'caption\' in arms\n    rows = read_jsonl(Path(args.split) / \'train.jsonl\')\n    evidence = read_jsonl(args.evidence)\n    by_id = {r[\'video_id\']: r for r in evidence}\n    if len(by_id) != len(evidence):\n        raise ValueError(\'Duplicate evidence; do not select the best retry\')\n    if set(by_id) != {r[\'video_id\'] for r in rows}:\n        raise ValueError(\'Evidence must cover exactly the frozen train IDs, including failures\')\n    accepted, rejected = [], []\n    for row in rows:\n        e = by_id[row[\'video_id\']]\n        try:\n            identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            if e.get(\'input_sha256\') != digest(identity):\n                raise ValueError(\'Evidence input identity mismatch\')\n            if need_graph:\n                validate_graph(e[\'stsg\'], row[\'video_duration\'])\n            if need_caption and (set(e[\'caption\']) != {\'caption\'} or not isinstance(e[\'caption\'][\'caption\'], str) or not e[\'caption\'][\'caption\'].strip()):\n                raise ValueError(\'Invalid caption\')\n            accepted.append(row)\n        except (KeyError, ValueError, TypeError) as exc:\n            rejected.append({\'video_id\': row[\'video_id\'], \'error\': str(exc)})\n    if not accepted:\n        raise ValueError(\'No common valid evidence; inspect teacher outputs\')\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=False)\n    for arm in arms:\n        write_jsonl(out / (arm + \'.jsonl\'), [s for r in accepted for s in sft_rows(r, by_id[r[\'video_id\']], arm)])\n    (out / \'build.json\').write_text(json.dumps(dict(accepted=len(accepted), rejected=rejected,\n        arms=list(arms), source_count=len(rows), evidence_sha256=digest(evidence), ids=[r[\'video_id\'] for r in accepted],\n        note=\'Same accepted video IDs and two rows per video in every arm. Schema validation is not factual verification.\'), indent=2), encoding=\'utf-8\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    p = sub.add_parser(\'split\')\n    for flag in (\'train\', \'test\', \'video-root\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.add_argument(\'--train-size\', type=int, default=256)\n    p.add_argument(\'--dev-size\', type=int, default=64)\n    p.add_argument(\'--seed\', type=int, default=20260923)\n    p.set_defaults(run=split)\n    p = sub.add_parser(\'build\')\n    p.add_argument(\'--arms\', nargs=\'+\', choices=ARMS, default=list(ARMS))\n    for flag in (\'split\', \'evidence\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.set_defaults(run=build)\n    args = parser.parse_args()\n    args.run(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/run.py': '"""Generate train-only pseudo graphs/captions, or evaluate a fresh SFT adapter."""\nimport argparse\nimport importlib.metadata\nimport json\nimport time\nfrom pathlib import Path\n\nfrom core import ROOT, dar_prompt, digest, evidence_prompt, parse_json, read_jsonl, validate_graph\n\n\nclass Backend:\n    def __init__(self, model, adapter=None, frames=16, pixels=100352):\n        import torch\n        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\n        from qwen_vl_utils import process_vision_info\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'This pilot needs a CUDA GPU; use the Kaggle notebook\')\n        self.torch, self.vision = torch, process_vision_info\n        self.processor = AutoProcessor.from_pretrained(model)\n        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n            model, torch_dtype=torch.float16, device_map=\'auto\', attn_implementation=\'sdpa\')\n        if adapter:\n            from peft import PeftModel\n            self.model = PeftModel.from_pretrained(self.model, adapter)\n        self.model.eval()\n        self.frames, self.pixels = frames, pixels\n\n    def generate(self, row, prompt, max_tokens):\n        if not Path(row[\'video_path\']).is_file():\n            raise FileNotFoundError(row[\'video_path\'])\n        messages = [{\'role\': \'user\', \'content\': [\n            {\'type\': \'video\', \'video\': row[\'video_path\'], \'nframes\': self.frames,\n             \'min_pixels\': self.pixels, \'max_pixels\': self.pixels},\n            {\'type\': \'text\', \'text\': prompt}]}]\n        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        # Qwen2.5 needs sampled fps, not source fps. The local utility supplies it\n        # when return_video_metadata=False. Do not drop these video_kwargs.\n        # False is the utility\'s default; omitting the keyword also supports\n        # pre-Qwen3 versions which already return the required sampled fps.\n        images, videos, kwargs = self.vision(messages, return_video_kwargs=True)\n        if not kwargs.get(\'fps\'):\n            raise RuntimeError(\'Qwen2.5 video input is missing sampled fps\')\n        frame_hash = digest([self.frames, self.pixels])\n        if videos:\n            import hashlib\n            frame_hash = hashlib.sha256(videos[0].numpy().tobytes()).hexdigest()\n        inputs = self.processor(text=[text], images=images, videos=videos, padding=True,\n                                return_tensors=\'pt\', **kwargs).to(self.model.device)\n        self.torch.cuda.synchronize()\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)\n        self.torch.cuda.synchronize()\n        tokens = output[:, inputs.input_ids.shape[1]:]\n        result = dict(raw=self.processor.batch_decode(tokens, skip_special_tokens=True)[0],\n                      generated_tokens=tokens.shape[1], input_tokens=inputs.input_ids.shape[1],\n                      generation_seconds=time.monotonic() - started, frame_sha256=frame_hash,\n                      sampled_fps=kwargs.get(\'fps\'), prompt_sha256=digest(prompt),\n                      hit_token_limit=tokens.shape[1] >= max_tokens)\n        del inputs, output, tokens\n        return result\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'mode\', choices=[\'teacher\', \'predict\'])\n    for flag in (\'model\', \'input\', \'output\'):\n        p.add_argument(\'--\' + flag, required=True)\n    p.add_argument(\'--adapter\')\n    p.add_argument(\'--backend\', choices=[\'qwen\', \'videollama3\'], default=\'qwen\')\n    p.add_argument(\'--kinds\', nargs=\'+\', choices=[\'caption\', \'stsg\'], default=[\'caption\', \'stsg\'])\n    p.add_argument(\'--load-in-4bit\', action=\'store_true\')\n    p.add_argument(\'--frames\', type=int, default=16)\n    p.add_argument(\'--max-tokens\', type=int, default=2048)\n    p.add_argument(\'--seed\', type=int, default=1234)\n    args = p.parse_args()\n    out = Path(args.output)\n    if out.exists() or out.with_suffix(\'.meta.json\').exists():\n        raise FileExistsError(\'Choose a new output; no silent retries or overwrites\')\n    rows = read_jsonl(args.input)\n    if not rows or len({r[\'video_id\'] for r in rows}) != len(rows):\n        raise ValueError(\'Empty manifest or duplicate video IDs\')\n    if args.mode == \'teacher\' and any(set(r) != {\'video_id\', \'video_path\', \'video_duration\'} for r in rows):\n        raise ValueError(\'Teacher input must be label-free; use teacher_inputs.jsonl\')\n    from transformers import set_seed\n    set_seed(args.seed)\n    if args.backend == \'videollama3\':\n        if args.mode != \'teacher\' or args.adapter:\n            raise ValueError(\'VideoLLaMA3 is a frozen teacher only\')\n        from teacher_videollama3 import VideoLLaMA3Backend\n        backend = VideoLLaMA3Backend(args.model, args.frames, args.load_in_4bit)\n    else:\n        if args.load_in_4bit:\n            raise ValueError(\'4-bit is currently implemented for VideoLLaMA3 teacher only\')\n        backend = Backend(args.model, args.adapter, args.frames)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    versions = {x: importlib.metadata.version(x) for x in (\'torch\', \'transformers\', \'accelerate\')}\n    meta = dict(config=vars(args), input_sha256=digest(rows), packages=versions,\n                code_sha256={p.name: digest(p.read_text(encoding=\'utf-8\')) for p in [Path(__file__), Path(__file__).with_name(\'core.py\'), ROOT / \'test.py\']})\n    if args.backend == \'videollama3\':\n        meta[\'teacher\'] = backend.metadata\n        backend_path = Path(__file__).with_name(\'teacher_videollama3.py\')\n        meta[\'code_sha256\'][backend_path.name] = digest(backend_path.read_text(encoding=\'utf-8\'))\n    out.with_suffix(\'.meta.json\').write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\n    with out.open(\'x\', encoding=\'utf-8\') as stream:\n        for i, row in enumerate(rows):\n            identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            result = dict(video_id=row[\'video_id\'], input_sha256=digest(identity), calls={})\n            for kind in (args.kinds if args.mode == \'teacher\' else (\'dar\',)):\n                prompt = dar_prompt(row[\'video_duration\']) if kind == \'dar\' else evidence_prompt(row[\'video_duration\'], kind)\n                try:\n                    call = backend.generate(identity, prompt, args.max_tokens)\n                    result[\'calls\'][kind] = call\n                    obj = parse_json(call[\'raw\'])\n                    if kind == \'stsg\':\n                        validate_graph(obj, row[\'video_duration\'])\n                    result[kind] = obj\n                except (ValueError, KeyError, TypeError) as exc:\n                    result[kind + \'_error\'] = str(exc)\n                # OOM, corrupt/missing video and model errors abort: do not silently\n                # turn a broken runtime into an apparent poor benchmark result.\n            stream.write(json.dumps(result, ensure_ascii=False, allow_nan=False) + \'\\n\')\n            stream.flush()\n            print(f\'{i + 1}/{len(rows)} {row["video_id"]}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/evaluate.py': '"""Paired, video-level evaluation. Missing/invalid outputs count as failures."""\nimport argparse\nimport copy\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nfrom core import digest, number, official_functions, read_jsonl\n\n\nOFFICIAL = official_functions()\n\n\ndef valid_segments(obj, duration):\n    if not isinstance(obj, dict) or set(obj) != {\'segments\'}:\n        return []\n    segs = obj[\'segments\']\n    if not isinstance(segs, list) or not segs:\n        return []\n    end, emotion = 0., None\n    try:\n        for s in segs:\n            start, stop = number(s[\'start_time\']), number(s[\'end_time\'])\n            if abs(start - end) > .051 or stop <= start or stop > duration + .051:\n                return []\n            if any(abs(x - round(x, 1)) > 1e-6 for x in (start, stop)):\n                return []\n            if s[\'emotion\'] not in OFFICIAL[\'EMOTION_CANDIDATES\'] or s[\'emotion\'] == emotion:\n                return []\n            if not isinstance(s[\'reason\'], str) or not s[\'reason\'].strip():\n                return []\n            end, emotion = stop, s[\'emotion\']\n        return segs if abs(end - duration) <= .051 else []\n    except (ValueError, TypeError, KeyError):\n        return []\n\n\ndef stats(pred, gt):\n    matched = 0\n    if pred and gt:\n        eligible = np.array([[OFFICIAL[\'calculate_iou\'](a, b) >= .5 and a[\'emotion\'] == b[\'emotion\'] for b in gt] for a in pred])\n        i, j = linear_sum_assignment(eligible.astype(int), maximize=True)\n        matched = int(eligible[i, j].sum())\n    return [matched, len(pred), len(gt), int(bool(pred)), int(len(pred) == len(gt))]\n\n\ndef aggregate(array):\n    tp, p, g, valid, count = array.sum(axis=0)\n    return dict(joint_f1_at_05=float(2 * tp / (p + g)) if p + g else 0.,\n                coverage=float(valid / len(array)), count_accuracy=float(count / len(array)),\n                predicted_segments=int(p), gt_segments=int(g))\n\n\ndef evaluate(manifest, predictions):\n    by_id = {r[\'video_id\']: r for r in predictions}\n    if len(by_id) != len(predictions) or set(by_id) - {r[\'video_id\'] for r in manifest}:\n        raise ValueError(\'Duplicate or out-of-manifest predictions\')\n    arrays, official_results = [], {}\n    for row in manifest:\n        prediction = by_id.get(row[\'video_id\'], {})\n        identity = {k: row[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n        if prediction.get(\'input_sha256\') is not None and prediction[\'input_sha256\'] != digest(identity):\n            raise ValueError(\'Prediction is from a different video input manifest\')\n        obj = prediction.get(\'dar\')\n        pred = valid_segments(obj, row[\'video_duration\'])\n        gt = row[\'target\'][\'segments\']\n        arrays.append(stats(pred, gt))\n        # A separate, explicitly labelled compatibility view uses repo repairs.\n        try:\n            repaired = OFFICIAL[\'validate_and_fix_segments\'](copy.deepcopy(obj[\'segments\']), row[\'video_duration\']) if obj else []\n            result = OFFICIAL[\'evaluate_single_video\'](repaired, gt)\n            if not all(np.isfinite(v) for v in result[\'ious\']):\n                raise ValueError(\'Non-finite repaired times\')\n        except (ValueError, TypeError, KeyError, AttributeError):\n            result = OFFICIAL[\'evaluate_single_video\']([], gt)\n        official_results[row[\'video_id\']] = result\n    array = np.asarray(arrays)\n    return array, dict(strict=aggregate(array), official_repaired=OFFICIAL[\'compute_overall_metrics\'](official_results),\n                       missing_videos=len(manifest) - len(by_id))\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--manifest\', required=True)\n    p.add_argument(\'--predictions\', nargs=\'+\', required=True, help=\'name=path pairs\')\n    p.add_argument(\'--reference\', default=\'caption\')\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    args = p.parse_args()\n    manifest = read_jsonl(args.manifest)\n    if not manifest or len({r[\'video_id\'] for r in manifest}) != len(manifest):\n        raise ValueError(\'Empty or duplicate manifest\')\n    arrays, summary, frames = {}, {}, {}\n    for item in args.predictions:\n        name, path = item.split(\'=\', 1)\n        if name in arrays:\n            raise ValueError(\'Repeated arm name\')\n        predictions = read_jsonl(path)\n        for row in predictions:\n            h = row.get(\'calls\', {}).get(\'dar\', {}).get(\'frame_sha256\')\n            if h is not None and row[\'video_id\'] in frames and frames[row[\'video_id\']] != h:\n                raise ValueError(\'Mismatched sampled frames across arms\')\n            if h is not None:\n                frames[row[\'video_id\']] = h\n        arrays[name], summary[name] = evaluate(manifest, predictions)\n    if args.reference not in arrays or args.bootstrap < 1:\n        raise ValueError(\'Missing reference or invalid bootstrap count\')\n    contrasts = {}\n    for name, array in arrays.items():\n        if name == args.reference:\n            continue\n        rng = np.random.default_rng(20260923)\n        diffs = []\n        for _ in range(args.bootstrap):\n            idx = rng.integers(0, len(manifest), len(manifest))\n            diffs.append(aggregate(array[idx])[\'joint_f1_at_05\'] - aggregate(arrays[args.reference][idx])[\'joint_f1_at_05\'])\n        contrasts[name + \'_minus_\' + args.reference] = dict(\n            delta=summary[name][\'strict\'][\'joint_f1_at_05\'] - summary[args.reference][\'strict\'][\'joint_f1_at_05\'],\n            paired_video_bootstrap_95ci=np.quantile(diffs, [.025, .975]).tolist())\n    with Path(args.output).open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(dict(videos=len(manifest), arms=summary, contrasts=contrasts,\n                       note=\'Fractions, not percentages. CI conditions on training seed and selected videos; not training variance.\'), stream, indent=2)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'experiments/dar_stsg/train.sh': '#!/usr/bin/env bash\n# Fresh full SFT per arm: frozen vision, train LLM + aligner, like DAR baseline.\nset -euo pipefail\n: "${BASE_MODEL:?Set BASE_MODEL to the local Qwen2.5-VL-3B-Instruct directory}"\n: "${DATA_DIR:?Set DATA_DIR to prepare.py build output}"\n: "${OUTPUT_ROOT:?Set OUTPUT_ROOT to a fresh output directory}"\nARM="${ARM:-stsg}"\ncase "$ARM" in baseline|caption|stsg|stsg_no_links) ;; *) exit 2 ;; esac\nSCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"\nREPO_ROOT="$(cd "$SCRIPT_DIR/../.." && pwd)"\n# Use the installed, version-pinned runtime; do not shadow it with the repo\'s\n# independent 4.0 development checkout when using the Kaggle 3.12.5 wheelhouse.\nexport CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES:-0}"\nexport FPS_MIN_FRAMES=16 FPS_MAX_FRAMES=16 VIDEO_MIN_PIXELS=100352 VIDEO_MAX_PIXELS=100352\nexport VIDEO_MIN_TOKEN_NUM=128 VIDEO_MAX_TOKEN_NUM=128\nexport TOKENIZERS_PARALLELISM=false\nSEED="${SEED:-1234}"\nDEST="$OUTPUT_ROOT/$ARM-seed$SEED"\nif [[ -e "$DEST" ]]; then echo "Output already exists: $DEST" >&2; exit 2; fi\npython -m swift.cli.sft \\\n  --model "$BASE_MODEL" --model_type qwen2_5_vl --template qwen2_5_vl \\\n  --dataset "$DATA_DIR/$ARM.jsonl" --split_dataset_ratio 0 \\\n  --train_type full --freeze_vit true --freeze_llm false --freeze_aligner false \\\n  --torch_dtype bfloat16 --fp16 false --bf16 true --attn_impl sdpa \\\n  --max_length 8192 --truncation_strategy delete --strict true \\\n  --num_train_epochs "${NUM_TRAIN_EPOCHS:-0.5}" --learning_rate "${LEARNING_RATE:-1e-5}" \\\n  --optim adamw_torch --weight_decay 0 \\\n  --per_device_train_batch_size 1 --gradient_accumulation_steps "${GRADIENT_ACCUMULATION_STEPS:-32}" \\\n  --gradient_checkpointing true --warmup_ratio 0.03 --lr_scheduler_type cosine \\\n  --seed "$SEED" --data_seed "$SEED" --dataloader_num_workers 0 \\\n  --save_strategy epoch --save_total_limit 1 --save_only_model true \\\n  --logging_steps 1 --report_to none --output_dir "$DEST"\n', 'experiments/dar_stsg/test_pipeline.py': 'import copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\n\nfrom core import annotation, digest, evidence_prompt, official_functions, sft_rows, validate_graph, write_jsonl, read_jsonl\nfrom evaluate import aggregate, evaluate, stats, valid_segments\nfrom prepare import build, split\n\n\ndef graph():\n    return {\'entities\': [{\'id\': \'o1\', \'label\': \'ball\'}, {\'id\': \'o2\', \'label\': \'table\'}],\n            \'events\': [{\'id\': \'e1\', \'start_time\': 0., \'end_time\': 1., \'observation\': \'Ball on table\', \'relations\': [[\'o1\', \'on\', \'o2\']]},\n                       {\'id\': \'e2\', \'start_time\': 1., \'end_time\': 2., \'observation\': \'Ball rolls\', \'relations\': []}],\n            \'temporal_links\': [[\'e1\', \'before\', \'e2\']]}\n\n\ndef segment(a=0., b=2., emotion=\'Interest\'):\n    return dict(start_time=a, end_time=b, emotion=emotion, reason=\'Visible object attracts attention.\')\n\n\ndef row(i=\'v\'):\n    return dict(video_id=i, video_path=i + \'.mp4\', video_duration=2., target={\'segments\': [segment()]})\n\n\nclass PipelineTests(unittest.TestCase):\n    def test_graph_prompt_has_no_copyable_placeholder_json(self):\n        prompt = evidence_prompt(2, \'stsg\')\n        self.assertNotIn(\'"o1"\', prompt)\n        self.assertNotIn(\'"e1"\', prompt)\n        self.assertNotIn(\'visible object or scene\', prompt)\n        self.assertIn(\'Both IDs MUST occur\', prompt)\n\n    def test_invalid_references_and_time(self):\n        self.assertEqual(validate_graph(graph(), 2), graph())\n        for change in (\'id\', \'time\', \'nan\', \'reverse\'):\n            g = graph()\n            if change == \'id\': g[\'events\'][0][\'relations\'][0][2] = \'missing\'\n            if change == \'time\': g[\'events\'][0][\'end_time\'] = 3\n            if change == \'nan\': g[\'events\'][0][\'start_time\'] = float(\'nan\')\n            if change == \'reverse\': g[\'temporal_links\'] = [[\'e2\', \'before\', \'e1\']]\n            with self.assertRaises(ValueError): validate_graph(g, 2)\n\n    def test_auxiliary_does_not_receive_answer(self):\n        r = row()\n        r[\'target\'][\'segments\'][0][\'reason\'] = \'SECRET_REFERENCE_TEXT\'\n        e = dict(stsg=graph(), caption={\'caption\': \'A ball rolls.\'})\n        for arm in (\'caption\', \'stsg\', \'stsg_no_links\'):\n            samples = sft_rows(r, e, arm)\n            self.assertIn(\'SECRET_REFERENCE_TEXT\', samples[0][\'messages\'][1][\'content\'])\n            self.assertNotIn(\'SECRET_REFERENCE_TEXT\', json.dumps(samples[1]))\n            self.assertEqual(len(samples), 2)\n        self.assertEqual(graph(), e[\'stsg\'])\n        no_links = json.loads(sft_rows(r, e, \'stsg_no_links\')[1][\'messages\'][1][\'content\'])\n        self.assertEqual(no_links[\'temporal_links\'], [])\n        self.assertEqual(no_links[\'events\'], e[\'stsg\'][\'events\'])\n\n    def test_no_gt_duration_fallback(self):\n        with self.assertRaises(ValueError): annotation({\'video\': \'v.mp4\', \'conversations\': []})\n\n    def test_full_split_uses_every_non_dev_train_video(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            source = lambda i: dict(video=f\'{i}.mp4\', video_duration=2.,\n                                    conversations=[{\'from\': \'gpt\', \'value\': json.dumps(row()[\'target\'])}])\n            write_jsonl(root / \'train.jsonl\', [source(i) for i in range(6)])\n            write_jsonl(root / \'test.jsonl\', [source(99)])\n            split(SimpleNamespace(train=root / \'train.jsonl\', test=root / \'test.jsonl\',\n                                  video_root=tmp, output=root / \'split\', train_size=-1,\n                                  dev_size=2, seed=1))\n            train = read_jsonl(root / \'split/train.jsonl\')\n            dev = read_jsonl(root / \'split/dev.jsonl\')\n            self.assertEqual((len(train), len(dev)), (4, 2))\n            self.assertEqual(len({r[\'video_id\'] for r in train + dev}), 6)\n\n    def test_matching_penalizes_extra_segments(self):\n        gt = [segment()]\n        self.assertEqual(stats(gt * 2, gt)[0], 1)\n        self.assertAlmostEqual(aggregate(np.asarray([stats(gt * 2, gt)]))[\'joint_f1_at_05\'], 2/3)\n        self.assertEqual(stats([segment(emotion=\'Fear\')], gt)[0], 0)\n\n    def test_invalid_outputs_and_missing_denominator(self):\n        self.assertEqual(valid_segments({\'segments\': [segment(b=float(\'nan\'))]}, 2), [])\n        self.assertEqual(valid_segments({\'segments\': [segment(a=1.)]}, 2), [])\n        a, report = evaluate([row(\'a\'), row(\'b\')], [{\'video_id\': \'a\', \'dar\': row()[\'target\']}])\n        self.assertEqual(report[\'strict\'][\'coverage\'], .5)\n        self.assertEqual(report[\'missing_videos\'], 1)\n        self.assertEqual(report[\'strict\'][\'gt_segments\'], 2)\n        self.assertAlmostEqual(report[\'strict\'][\'joint_f1_at_05\'], 2/3)\n\n    def test_duplicate_predictions_fail(self):\n        p = {\'video_id\': \'v\', \'dar\': row()[\'target\']}\n        with self.assertRaises(ValueError): evaluate([row()], [p, p])\n\n    def test_official_functions_preserved(self):\n        f = official_functions()\n        self.assertIn(\'SILENT\', f[\'build_eval_prompt\'](2.))\n        self.assertEqual(f[\'evaluate_single_video\']([segment()], [segment()])[\'avg_iou\'], 1.)\n\n    def test_split_and_common_support(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            def source(i):\n                return dict(video=f\'{i}.mp4\', video_duration=2., conversations=[{\'from\': \'gpt\', \'value\': json.dumps(row()[\'target\'])}])\n            write_jsonl(root/\'train.jsonl\', [source(i) for i in range(6)])\n            write_jsonl(root/\'test.jsonl\', [source(99)])\n            args = SimpleNamespace(train=root/\'train.jsonl\', test=root/\'test.jsonl\', video_root=tmp,\n                                   output=root/\'split\', train_size=3, dev_size=2, seed=1)\n            split(args)\n            inputs = read_jsonl(root/\'split/teacher_inputs.jsonl\')\n            self.assertTrue(all(set(r) == {\'video_id\', \'video_path\', \'video_duration\'} for r in inputs))\n            evidence = [dict(video_id=r[\'video_id\'], input_sha256=digest(r), stsg=graph(), caption={\'caption\': \'Ball\'}) for r in inputs]\n            evidence[0].pop(\'stsg\')\n            write_jsonl(root/\'evidence.jsonl\', evidence)\n            build(SimpleNamespace(split=root/\'split\', evidence=root/\'evidence.jsonl\', output=root/\'arms\'))\n            for arm in (\'baseline\', \'caption\', \'stsg\', \'stsg_no_links\'):\n                self.assertEqual(len(read_jsonl(root/\'arms\'/f\'{arm}.jsonl\')), 4)\n            write_jsonl(root/\'overlap.jsonl\', [source(1)])\n            args.test = root/\'overlap.jsonl\'\n            args.output = root/\'bad\'\n            with self.assertRaises(ValueError): split(args)\n\n    def test_single_arm_only_requires_its_evidence(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            r = row()\n            write_jsonl(root / \'train.jsonl\', [r])\n            identity = {k: r[k] for k in (\'video_id\', \'video_path\', \'video_duration\')}\n            for arm, payload in [(\'stsg\', {\'stsg\': graph()}),\n                                 (\'caption\', {\'caption\': {\'caption\': \'A ball rolls.\'}})]:\n                evidence_path = root / (arm + \'-evidence.jsonl\')\n                write_jsonl(evidence_path, [dict(video_id=r[\'video_id\'], input_sha256=digest(identity), **payload)])\n                output = root / arm\n                build(SimpleNamespace(split=root, evidence=evidence_path, output=output, arms=[arm]))\n                self.assertEqual(len(read_jsonl(output / (arm + \'.jsonl\'))), 2)\n                self.assertEqual([p.name for p in output.glob(\'*.jsonl\')], [arm + \'.jsonl\'])\n                other = \'caption\' if arm == \'stsg\' else \'stsg\'\n                with self.assertRaises(ValueError):\n                    build(SimpleNamespace(split=root, evidence=evidence_path, output=root/\'invalid\', arms=[other]))\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'experiments/dar_stsg/teacher_videollama3.py': '"""Frozen VideoLLaMA3 teacher; run in a separate Transformers 4.57.1 process.\n\nAPI: https://github.com/DAMO-NLP-SG/VideoLLaMA3/blob/main/inference/example_videollama3.py\nUse a full local snapshot including custom Python files, not the Image checkpoint.\n"""\nimport hashlib\nimport importlib.metadata\nimport time\nfrom pathlib import Path\n\nfrom core import digest\n\n\nclass VideoLLaMA3Backend:\n    def __init__(self, model, frames=16, load_in_4bit=False):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig\n        from transformers import image_utils\n        if transformers.__version__ != \'4.57.1\':\n            raise RuntimeError(\n                \'Use the separate teacher runtime: transformers==4.57.1; \'\n                f\'loaded {transformers.__version__} from {transformers.__file__}. \'\n                \'Rerun the offline teacher bootstrap and use its teacher_python/teacher_env.\')\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'VideoLLaMA3 teacher requires CUDA\')\n        # The snapshot uses VideoInput only as a type annotation. Transformers\n        # 4.57.1 no longer exports it, while its ImageInput covers the same\n        # accepted image/frame values used by this processor.\n        if not hasattr(image_utils, \'VideoInput\'):\n            image_utils.VideoInput = image_utils.ImageInput\n        root = Path(model)\n        if not root.is_dir() or not list(root.glob(\'modeling*.py\')):\n            raise ValueError(\'Attach full local VideoLLaMA3 snapshot, including custom Python code\')\n        self.torch, self.frames = torch, frames\n        options = dict(trust_remote_code=True, local_files_only=True,\n                       torch_dtype=torch.float16, device_map=\'auto\', attn_implementation=\'sdpa\')\n        if load_in_4bit:\n            options[\'quantization_config\'] = BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n                llm_int8_skip_modules=[\'vision_encoder\', \'mm_projector\', \'lm_head\'])\n        else:\n            # Leave activation/KV-cache headroom on every visible GPU.\n            available = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]\n            if sum(available) < 22 * 1024**3:\n                raise RuntimeError(\'FP16 teacher needs more headroom: use two GPUs or --load-in-4bit\')\n            options[\'max_memory\'] = {i: int(n * .8) for i, n in enumerate(available)}\n        self.processor = AutoProcessor.from_pretrained(model, trust_remote_code=True, local_files_only=True)\n        self.model = AutoModelForCausalLM.from_pretrained(model, **options).eval()\n        self.metadata = dict(model_config=self.model.config.to_dict(),\n            snapshot_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest()\n                             for p in sorted(root.iterdir()) if p.suffix in (\'.py\', \'.json\')},\n            device_map={k: str(v) for k, v in getattr(self.model, \'hf_device_map\', {}).items()},\n            precision=\'nf4-fp16\' if load_in_4bit else \'fp16\',\n            sampling=\'uniform up to max_frames; native VideoLLaMA3 preprocessing\')\n        if load_in_4bit:\n            self.metadata[\'bitsandbytes\'] = importlib.metadata.version(\'bitsandbytes\')\n\n    def generate(self, row, prompt, max_tokens):\n        import numpy as np\n        from decord import VideoReader, cpu\n        if not Path(row[\'video_path\']).is_file():\n            raise FileNotFoundError(row[\'video_path\'])\n        reader = VideoReader(row[\'video_path\'], ctx=cpu(0), num_threads=2)\n        fps = float(reader.get_avg_fps())\n        if len(reader) < 1 or not np.isfinite(fps) or fps <= 0:\n            raise RuntimeError(\'Video has no frames or invalid FPS\')\n        indices = np.linspace(0, len(reader) - 1, min(self.frames, len(reader)), dtype=int)\n        frames = reader.get_batch(indices).asnumpy().transpose(0, 3, 1, 2)\n        timestamps = (indices / fps).tolist()\n        conversation = [{\'role\': \'user\', \'content\': [\n            {\'type\': \'video\', \'video\': list(frames), \'num_frames\': len(frames),\n             \'timestamps\': timestamps},\n            {\'type\': \'text\', \'text\': prompt}]}]\n        inputs = self.processor(conversation=conversation, add_system_prompt=True,\n                                add_generation_prompt=True, return_tensors=\'pt\')\n        pixel_hash = hashlib.sha256(inputs[\'pixel_values\'].contiguous().numpy().tobytes()).hexdigest()\n        device = self.model.get_input_embeddings().weight.device\n        inputs = {k: v.to(device) if isinstance(v, self.torch.Tensor) else v for k, v in inputs.items()}\n        inputs[\'pixel_values\'] = inputs[\'pixel_values\'].to(self.torch.float16)\n        self.torch.cuda.synchronize()\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)\n        self.torch.cuda.synchronize()\n        # Custom generate uses inputs_embeds: output contains generated tokens only.\n        return dict(raw=self.processor.batch_decode(output, skip_special_tokens=True)[0].strip(),\n                    generated_tokens=output.shape[1], input_tokens=inputs[\'input_ids\'].shape[1],\n                    generation_seconds=time.monotonic() - started,\n                    visual_tensor_sha256=pixel_hash, prompt_sha256=digest(prompt),\n                    max_frames=self.frames, sampling=\'uniform\', frame_indices=indices.tolist(),\n                    timestamps=timestamps, source_fps=fps,\n                    hit_token_limit=output.shape[1] >= max_tokens)\n', 'test.py': '#!/usr/bin/env python\n"""\nInference and evaluation for DAR video emotion segmentation.\n"""\n\nimport argparse\nimport json\nimport gc\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Set\n\nfrom tqdm import tqdm\n\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")\nos.environ.setdefault("VLLM_USE_V1", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "true")\n\nimport torch\nfrom vllm import LLM, SamplingParams\nfrom transformers import AutoProcessor\n\nREPO_ROOT = Path(__file__).resolve().parent\nLOCAL_QWEN_VL_UTILS = REPO_ROOT / "qwen-vl-utils" / "src"\nif LOCAL_QWEN_VL_UTILS.exists():\n    sys.path.insert(0, str(LOCAL_QWEN_VL_UTILS))\nfrom qwen_vl_utils import process_vision_info\n\nEMOTION_CANDIDATES = [\n    "Awkwardness", "Empathic Pain", "Fear", "Anger", "Sadness", "Relief",\n    "Boredom", "Joy", "Aesthetic Appreciation", "Adoration", "Admiration",\n    "Amusement", "Satisfaction", "Disgust", "Sexual Desire", "Confusion",\n    "Romance", "Craving", "Horror", "Excitement", "Nostalgia",\n    "Awe (or Wonder)", "Interest", "Calmness", "Surprise", "Entrancement", "Anxiety"\n]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model-path", default=os.environ.get("MODEL_PATH", "/path/to/DAR-R1"))\n    parser.add_argument("--test-jsonl", default=os.environ.get("TEST_JSONL", "/path/to/DAR/test.jsonl"))\n    parser.add_argument("--output-jsonl", default=os.environ.get("OUT_JSONL", "/path/to/outputs/dar_r1_test_predictions.jsonl"))\n    parser.add_argument("--video-root", default=os.environ.get("VIDEO_ROOT", "/path/to/DAR/videos"))\n    parser.add_argument("--video-path-prefix", default=os.environ.get("VIDEO_PATH_PREFIX", ""))\n    parser.add_argument("--batch-size", type=int, default=int(os.environ.get("BATCH_SIZE", "4")))\n    parser.add_argument("--max-tokens", type=int, default=int(os.environ.get("MAX_TOKENS", "4096")))\n    parser.add_argument("--max-model-len", type=int, default=int(os.environ.get("MAX_MODEL_LEN", "16384")))\n    parser.add_argument("--gpu-memory-utilization", type=float, default=float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.8")))\n    parser.add_argument("--seed", type=int, default=int(os.environ.get("SEED", "1234")))\n    return parser.parse_args()\n\n\ndef load_test_data(test_jsonl: str) -> List[Dict[str, Any]]:\n    data = []\n    with open(test_jsonl, \'r\', encoding=\'utf-8\') as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                item = json.loads(line)\n                video_path = item.get("video", "")\n                video_duration = item.get("video_duration")\n                video_id = os.path.splitext(os.path.basename(video_path))[0]\n\n                gt_segments = []\n                conversations = item.get("conversations", [])\n                if len(conversations) >= 2:\n                    gpt_text = conversations[1].get("value", "")\n                    try:\n                        gt_parsed = json.loads(gpt_text)\n                        gt_segments = gt_parsed.get("segments", [])\n                    except json.JSONDecodeError:\n                        pass\n\n                data.append({\n                    "video_id": video_id,\n                    "video_path": video_path,\n                    "video_duration": video_duration if video_duration else 30.0,\n                    "gt_segments": gt_segments\n                })\n            except json.JSONDecodeError:\n                continue\n    return data\n\n\ndef resolve_video_path(video_path: str, video_root: str, video_path_prefix: str) -> str:\n    if os.path.exists(video_path):\n        return video_path\n    if video_path_prefix and video_path.startswith(video_path_prefix):\n        return os.path.join(video_root, video_path[len(video_path_prefix):])\n    if not os.path.isabs(video_path):\n        return os.path.join(video_root, video_path)\n    return video_path\n\n\ndef load_done_ids(out_jsonl: str) -> Set[str]:\n    done = set()\n    if not os.path.exists(out_jsonl):\n        return done\n    with open(out_jsonl, "r", encoding="utf-8") as f:\n        for line in f:\n            try:\n                j = json.loads(line)\n                vid = str(j.get("video_id", ""))\n                if vid and j.get("segments"):\n                    done.add(vid)\n            except Exception:\n                continue\n    return done\n\n\ndef build_eval_prompt(video_duration: float) -> str:\n    cand_str = ", ".join(EMOTION_CANDIDATES)\n    dur_str = f"{video_duration:.1f}" if video_duration and video_duration > 0 else "unknown"\n\n    prompt = f"""You are analyzing a **SILENT video** (no audio/music/dialogue) to understand the **viewer\'s emotional journey**.\n\n## Your Task\n1. **Segment** the video into emotion phases based on when the viewer\'s dominant emotion CHANGES\n2. **Predict** the viewer\'s emotion for each segment (must be from the allowed list)\n3. **Reason** why the viewer feels that emotion\n\n## Core Principles (Based on Appraisal-Event Theory)\n- **Observation**: What visual elements do you see? (actions, objects, colors, lighting, composition)\n- **Causality**: Why would these specific visual elements trigger a particular emotion in viewers?\n- **Dynamics**: If the emotion changes from the previous segment, what specific event caused this transition?\n- **Adaptive Merging**: Emotional responses persist until a new event triggers change. Therefore:\n  - **CRITICAL: Adjacent segments MUST have DIFFERENT emotions**\n  - If two consecutive time periods evoke the same emotion, they should be MERGED into ONE segment\n  - Only create a new segment when the viewer\'s emotion genuinely CHANGES\n\n## Viewer-Centered Focus\n- This is about the **VIEWER\'s emotion** while watching, NOT the emotions of characters in the video\n- Don\'t just describe what characters feel (e.g., "the man looks happy")\n- Explain what makes the VIEWER feel a certain way\n\n## Allowed Emotions (STRICTLY choose from these 27 only):\n{cand_str}\n\n## Hard Constraints\n1. Segments MUST cover the full video continuously:\n   - First segment starts at 0.0\n   - Segments are continuous: segment[i].end_time == segment[i+1].start_time\n   - Last segment ends at {dur_str} (video duration)\n2. Timestamps: seconds with 1 decimal place\n3. **CRITICAL: Adjacent segments MUST have DIFFERENT emotions** (merge if same)\n\n## Quality Criteria for reason:\n1. **Visual Grounding**: Reference visual cues that ACTUALLY exist in the video frames. Don\'t invent details.\n2. **Causal Logic**: Your reasoning must logically connect visual evidence to emotion.\n3. **Viewer-Centeredness**: Focus on the VIEWER\'s feelings, not just describing characters\' facial expressions.\n4. **Temporal Consistency**: Your reasoning should be consistent with the historical context (previous segments).\n\nOutput ONLY a JSON object with this schema:\n{{\n  "segments": [\n    {{\n      "start_time": 0.0,\n      "end_time": <float>,\n      "emotion": "<one of the 27 emotions>",\n      "reason": "<viewer-centered, visually grounded reason>"\n    }}, ...\n  ]\n}}"""\n    return prompt\n\n\ndef make_messages(video_path: str, video_duration: float) -> List[Dict[str, Any]]:\n    abs_vp = os.path.abspath(video_path)\n    prompt_text = build_eval_prompt(video_duration)\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "video", "video": abs_vp},\n            {"type": "text", "text": prompt_text}\n        ]\n    }]\n\n\ndef build_input(processor, messages):\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    image_inputs, video_inputs, video_kwargs = process_vision_info(\n        messages,\n        image_patch_size=processor.image_processor.patch_size,\n        return_video_kwargs=True,\n        return_video_metadata=True\n    )\n    mm_data = {}\n    if image_inputs is not None:\n        mm_data[\'image\'] = image_inputs\n    if video_inputs is not None:\n        mm_data[\'video\'] = video_inputs\n    return {\'prompt\': text, \'multi_modal_data\': mm_data, \'mm_processor_kwargs\': video_kwargs}\n\n\ndef try_parse_json(text: str) -> Optional[dict]:\n    if not isinstance(text, str):\n        return None\n    s = text.strip()\n    if s.startswith("```"):\n        first_newline = s.find("\\n")\n        if first_newline != -1:\n            s = s[first_newline+1:]\n        if s.endswith("```"):\n            s = s[:-3]\n        s = s.strip()\n    l = s.find("{")\n    r = s.rfind("}")\n    if l == -1 or r == -1 or r <= l:\n        return None\n    try:\n        return json.loads(s[l:r+1])\n    except Exception:\n        return None\n\n\ndef merge_adjacent_same_emotion_segments(segments: List[Dict]) -> List[Dict]:\n    if not segments or len(segments) <= 1:\n        return segments\n    merged = []\n    current = segments[0].copy()\n    for i in range(1, len(segments)):\n        next_seg = segments[i]\n        if next_seg.get("emotion", "") == current.get("emotion", ""):\n            current["end_time"] = next_seg.get("end_time", current["end_time"])\n            cur_reason = current.get("reason", "")\n            nxt_reason = next_seg.get("reason", "")\n            if nxt_reason and nxt_reason not in cur_reason:\n                current["reason"] = cur_reason + " " + nxt_reason\n        else:\n            merged.append(current)\n            current = next_seg.copy()\n    merged.append(current)\n    return merged\n\n\ndef validate_and_fix_segments(segments: List[Dict], video_duration: float) -> List[Dict]:\n    if not segments:\n        return []\n    segments.sort(key=lambda x: x.get("start_time", 0))\n    for seg in segments:\n        emo = seg.get("emotion", "")\n        if emo not in EMOTION_CANDIDATES:\n            matched = False\n            for cand in EMOTION_CANDIDATES:\n                if cand.lower() in emo.lower() or emo.lower() in cand.lower():\n                    seg["emotion"] = cand\n                    matched = True\n                    break\n            if not matched:\n                seg["emotion"] = "Interest"\n    segments = merge_adjacent_same_emotion_segments(segments)\n    if segments[0].get("start_time", 0) != 0.0:\n        segments[0]["start_time"] = 0.0\n    if abs(segments[-1].get("end_time", 0) - video_duration) > 0.5:\n        segments[-1]["end_time"] = video_duration\n    for i in range(1, len(segments)):\n        if segments[i]["start_time"] != segments[i-1]["end_time"]:\n            segments[i]["start_time"] = segments[i-1]["end_time"]\n    return segments\n\n\ndef calculate_iou(seg1: Dict, seg2: Dict) -> float:\n    s1, e1 = seg1[\'start_time\'], seg1[\'end_time\']\n    s2, e2 = seg2[\'start_time\'], seg2[\'end_time\']\n    intersection = max(0, min(e1, e2) - max(s1, s2))\n    union = max(e1, e2) - min(s1, s2)\n    return intersection / union if union > 0 else 0.0\n\n\ndef evaluate_single_video(pred_segments: List[Dict], gt_segments: List[Dict]) -> Dict:\n    pred_count = len(pred_segments)\n    gt_count = len(gt_segments)\n    count_match = (pred_count == gt_count)\n\n    compare_count = min(pred_count, gt_count)\n\n    ious = []\n    emotion_matches = []\n    details = []\n\n    for i in range(compare_count):\n        pred_seg = pred_segments[i]\n        gt_seg = gt_segments[i]\n\n        iou = calculate_iou(pred_seg, gt_seg)\n        ious.append(iou)\n\n        pred_emo = pred_seg.get("emotion", "").strip()\n        gt_emo = gt_seg.get("emotion", "").strip()\n        emo_match = (pred_emo == gt_emo)\n        emotion_matches.append(emo_match)\n\n        details.append({\n            "segment_idx": i,\n            "pred_time": [pred_seg["start_time"], pred_seg["end_time"]],\n            "gt_time": [gt_seg["start_time"], gt_seg["end_time"]],\n            "iou": round(iou, 4),\n            "pred_emotion": pred_emo,\n            "gt_emotion": gt_emo,\n            "emotion_match": emo_match\n        })\n\n    return {\n        "count_match": count_match,\n        "pred_count": pred_count,\n        "gt_count": gt_count,\n        "compared": compare_count,\n        "avg_iou": sum(ious) / len(ious) if ious else 0.0,\n        "emotion_accuracy": sum(emotion_matches) / len(emotion_matches) if emotion_matches else 0.0,\n        "emotion_correct": sum(emotion_matches),\n        "ious": ious,\n        "emotion_matches": emotion_matches,\n        "details": details\n    }\n\n\ndef compute_overall_metrics(all_results: Dict[str, Dict]) -> Dict:\n    total_videos = len(all_results)\n    if total_videos == 0:\n        return {}\n\n    count_match_total = sum(1 for r in all_results.values() if r["count_match"])\n    all_ious = []\n    all_emotion_matches = []\n\n    for r in all_results.values():\n        all_ious.extend(r["ious"])\n        all_emotion_matches.extend(r["emotion_matches"])\n\n    return {\n        "total_videos": total_videos,\n        "segment_count_match": count_match_total,\n        "segment_count_match_rate": count_match_total / total_videos,\n        "total_compared_segments": len(all_ious),\n        "avg_iou": sum(all_ious) / len(all_ious) if all_ious else 0.0,\n        "emotion_accuracy": sum(all_emotion_matches) / len(all_emotion_matches) if all_emotion_matches else 0.0,\n        "emotion_correct": sum(all_emotion_matches),\n        "emotion_total": len(all_emotion_matches),\n    }\n\n\nif __name__ == "__main__":\n    args = parse_args()\n\n    print("=" * 80)\n    print("DAR video emotion segmentation inference and evaluation")\n    print("=" * 80)\n    print(f"Model: {args.model_path}")\n    print(f"Test JSONL: {args.test_jsonl}")\n    print(f"Output JSONL: {args.output_jsonl}")\n    print(f"Video root: {args.video_root}")\n    print(f"Batch size: {args.batch_size}")\n    print("=" * 80)\n\n    print("\\nLoading test data...")\n    test_data = load_test_data(args.test_jsonl)\n    print(f"Loaded {len(test_data)} test videos")\n\n    done_ids = load_done_ids(args.output_jsonl)\n    print(f"Completed videos in existing output: {len(done_ids)}")\n    videos_to_process = [v for v in test_data if v["video_id"] not in done_ids]\n    print(f"Videos to process: {len(videos_to_process)}")\n\n    if not videos_to_process:\n        print("\\nAll videos are already processed. Running evaluation.")\n    else:\n        print("\\nInitializing model...")\n        tensor_parallel_size = torch.cuda.device_count()\n        print(f"GPU count: {tensor_parallel_size}")\n\n        llm = LLM(\n            model=args.model_path,\n            trust_remote_code=True,\n            gpu_memory_utilization=args.gpu_memory_utilization,\n            tensor_parallel_size=tensor_parallel_size,\n            max_model_len=args.max_model_len,\n            seed=args.seed,\n        )\n        processor = AutoProcessor.from_pretrained(args.model_path)\n        sampling_params = SamplingParams(\n            temperature=0.1,\n            top_p=0.9,\n            max_tokens=args.max_tokens,\n            stop_token_ids=[],\n        )\n        print("Model initialized")\n\n        os.makedirs(os.path.dirname(args.output_jsonl) or \'.\', exist_ok=True)\n        fout = open(args.output_jsonl, "a", encoding="utf-8")\n\n        print("\\nStarting inference...")\n        success_count = 0\n        fail_count = 0\n\n        for batch_start in tqdm(range(0, len(videos_to_process), args.batch_size), desc="inference"):\n            batch_videos = videos_to_process[batch_start:batch_start + args.batch_size]\n            batch_inputs = []\n            batch_meta = []\n\n            for v in batch_videos:\n                vid = v["video_id"]\n                vp = resolve_video_path(v["video_path"], args.video_root, args.video_path_prefix)\n                if not os.path.exists(vp):\n                    result = {"video_id": vid, "video_path": vp, "segments": [], "error": "File not found"}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n                    continue\n                try:\n                    messages = make_messages(vp, v["video_duration"])\n                    inp = build_input(processor, messages)\n                    batch_inputs.append(inp)\n                    batch_meta.append(v)\n                except Exception as e:\n                    result = {"video_id": vid, "video_path": vp, "segments": [], "error": repr(e)}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n\n            if not batch_inputs:\n                continue\n\n            try:\n                outputs = llm.generate(batch_inputs, sampling_params=sampling_params)\n                for k, meta in enumerate(batch_meta):\n                    vid = meta["video_id"]\n                    duration = meta["video_duration"]\n                    try:\n                        raw_text = outputs[k].outputs[0].text\n                        parsed = try_parse_json(raw_text)\n                        if parsed and "segments" in parsed:\n                            segments = validate_and_fix_segments(parsed["segments"], duration)\n                            result = {"video_id": vid, "video_path": meta["video_path"],\n                                      "video_duration": duration, "segments": segments}\n                            success_count += 1\n                        else:\n                            result = {"video_id": vid, "video_path": meta["video_path"],\n                                      "video_duration": duration, "segments": [],\n                                      "error": "Parse failed", "raw_output": raw_text[:500]}\n                            fail_count += 1\n                    except Exception as e:\n                        result = {"video_id": vid, "video_path": meta["video_path"],\n                                  "segments": [], "error": repr(e)}\n                        fail_count += 1\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                del outputs\n                gc.collect()\n            except Exception as e:\n                for meta in batch_meta:\n                    result = {"video_id": meta["video_id"], "segments": [], "error": repr(e)}\n                    fout.write(json.dumps(result, ensure_ascii=False) + "\\n")\n                    fout.flush()\n                    fail_count += 1\n\n        fout.close()\n        print(f"\\nInference finished: success={success_count}, failed={fail_count}")\n\n    print("\\n" + "=" * 80)\n    print("Evaluation")\n    print("=" * 80)\n\n    gt_dict = {}\n    for item in test_data:\n        gt_dict[item["video_id"]] = item["gt_segments"]\n\n    pred_dict = {}\n    with open(args.output_jsonl, "r", encoding="utf-8") as f:\n        for line in f:\n            try:\n                j = json.loads(line)\n                vid = j.get("video_id", "")\n                segs = j.get("segments", [])\n                if vid and segs:\n                    pred_dict[vid] = segs\n            except Exception:\n                continue\n\n    print(f"GT videos: {len(gt_dict)}")\n    print(f"Predicted videos: {len(pred_dict)}")\n\n    common_ids = set(gt_dict.keys()) & set(pred_dict.keys())\n    print(f"Evaluable videos: {len(common_ids)}")\n\n    all_eval_results = {}\n    for vid in sorted(common_ids):\n        result = evaluate_single_video(pred_dict[vid], gt_dict[vid])\n        all_eval_results[vid] = result\n\n    metrics = compute_overall_metrics(all_eval_results)\n\n    print("\\n" + "=" * 80)\n    print("Evaluation Results")\n    print("=" * 80)\n    print(f"Evaluated videos:           {metrics.get(\'total_videos\', 0)}")\n    print(f"Segment count matches:      {metrics.get(\'segment_count_match\', 0)}")\n    print(f"Segment count match rate:   {metrics.get(\'segment_count_match_rate\', 0):.4f}")\n    print(f"Compared segments:          {metrics.get(\'total_compared_segments\', 0)}")\n    print(f"Mean temporal IoU:          {metrics.get(\'avg_iou\', 0):.4f}")\n    print(f"Emotion accuracy:           {metrics.get(\'emotion_accuracy\', 0):.4f} ({metrics.get(\'emotion_correct\', 0)}/{metrics.get(\'emotion_total\', 0)})")\n    print("=" * 80)\n\n    print("\\nEmotion accuracy under IoU thresholds:")\n    all_ious = []\n    all_emo_matches = []\n    for r in all_eval_results.values():\n        all_ious.extend(r["ious"])\n        all_emo_matches.extend(r["emotion_matches"])\n\n    for threshold in [0.3, 0.5, 0.7, 0.9]:\n        valid = [(iou, em) for iou, em in zip(all_ious, all_emo_matches) if iou >= threshold]\n        if valid:\n            acc = sum(em for _, em in valid) / len(valid)\n            print(f"  IOU >= {threshold:.1f}: {acc:.4f} ({sum(em for _, em in valid)}/{len(valid)} segments)")\n        else:\n            print(f"  IOU >= {threshold:.1f}: N/A (0 segments)")\n\n    print("\\nExample results (first 5 videos):")\n    for vid in sorted(common_ids)[:5]:\n        r = all_eval_results[vid]\n        print(f"\\n  video {vid}: pred={r[\'pred_count\']} gt={r[\'gt_count\']} "\n              f"count_match={r[\'count_match\']} "\n              f"avgIoU={r[\'avg_iou\']:.3f} emotion_acc={r[\'emotion_accuracy\']:.3f}")\n        for d in r["details"]:\n            print(f"    seg{d[\'segment_idx\']}: IOU={d[\'iou\']:.3f} "\n                  f"{d[\'pred_emotion\']} vs {d[\'gt_emotion\']} "\n                  f"match={d[\'emotion_match\']}")\n\n    eval_output_path = args.output_jsonl.replace(".jsonl", "_metrics.json")\n    eval_data = {\n        "summary": metrics,\n        "per_video": {vid: {k: v for k, v in r.items() if k != "details"} for vid, r in all_eval_results.items()}\n    }\n    with open(eval_output_path, "w", encoding="utf-8") as f:\n        json.dump(eval_data, f, ensure_ascii=False, indent=2)\n    print(f"\\nSaved metrics to: {eval_output_path}")\n'}
for name, content in FILES.items():
    dest = PROJECT / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        assert dest.read_text(encoding='utf-8') == content, f'Code khác version: {dest}'
    else:
        dest.write_text(content, encoding='utf-8')
SCRIPTS = PROJECT / 'experiments/dar_stsg'
WORK.mkdir(parents=True, exist_ok=True)
run(sys.executable, '-m', 'unittest', 'discover', '-s', SCRIPTS, '-p', 'test_pipeline.py', '-v')


Tách dev từ **train**, kiểm tra overlap với test. Không dùng official test để
chọn hyperparameters. File teacher chỉ chứa video ID/path/duration, không có target.
Lỗi file/decoder/GPU sẽ dừng; không lặng lẽ thay video.

In [ ]:
run(sys.executable, SCRIPTS / 'prepare.py', 'split', '--train', TRAIN_JSONL,
    '--test', TEST_JSONL, '--video-root', VIDEO_ROOT, '--train-size', TRAIN_SIZE,
    '--dev-size', DEV_SIZE, '--output', WORK / 'split')
for name in ('teacher_inputs.jsonl', 'dev.jsonl'):
    rows = [json.loads(x) for x in (WORK / 'split' / name).read_text().splitlines()]
    missing = [r['video_path'] for r in rows if not Path(r['video_path']).is_file()]
    assert not missing, f'VIDEO_ROOT phải chứa trực tiếp các basename: {missing[:5]}'


In [ ]:
kind = 'caption' if ARM == 'caption' else 'stsg'
teacher_command = [teacher_python, SCRIPTS / 'run.py', 'teacher', '--backend', 'videollama3',
    '--model', TEACHER_MODEL, '--kinds', kind, '--frames', '16',
    '--input', WORK / 'split/teacher_inputs.jsonl', '--output', WORK / 'evidence.jsonl']
if TEACHER_4BIT:
    teacher_command.append('--load-in-4bit')
subprocess.run(list(map(str, teacher_command)), check=True, env=teacher_env)
evidence = [json.loads(x) for x in (WORK / 'evidence.jsonl').read_text().splitlines()]
for item in evidence:
    call = item.get('calls', {}).get(kind, {})
    print('teacher evidence', item['video_id'],
          'error=', item.get(kind + '_error'),
          'parsed=', kind in item,
          'tokens=', call.get('generated_tokens'),
          'hit_limit=', call.get('hit_token_limit'),
          'raw_prefix=', repr(call.get('raw', '')[:600]), flush=True)
run(sys.executable, SCRIPTS / 'prepare.py', 'build', '--split', WORK / 'split',
    '--evidence', WORK / 'evidence.jsonl', '--output', WORK / 'arms', '--arms', ARM)
print((WORK / 'arms/build.json').read_text())
print(json.dumps(evidence[0], ensure_ascii=False, indent=2))


Trước pilot lớn, đối chiếu graph/caption của một mẫu train được chọn trước với
video. Schema pass chỉ kiểm tra cấu trúc. Chạy từng arm riêng có thể loại các video khác nhau;
đối chiếu accepted IDs trong build.json và cấu hình baseline trước khi so sánh. Nếu đa số
graph lỗi hoặc bịa sự kiện, cải thiện teacher/data trước khi tăng compute.

Train **full SFT** cho LLM + aligner, đóng băng vision encoder như baseline DAR.
BF16, LR=1e-5, AdamW, weight_decay=0, max_length=8192, gradient accumulation=32.
Train 0.5 epoch trên dataset DAR + auxiliary đã build, không giới hạn max_steps.
Pilot vẫn dùng tập con; mỗi video có 2 mẫu nên cùng số epoch chưa bảo đảm cùng ngân sách baseline.
Cần GPU hỗ trợ BF16 và đủ VRAM cho full SFT (baseline notebook ghi RTX 6000 96 GB).
Teacher 4-bit không làm student full SFT tốn ít VRAM hơn. Nếu OOM,
ghi lại cấu hình mới và chạy lại tất cả arm nhất quán; chưa xác nhận VRAM trên Kaggle.

In [ ]:
run(sys.executable, '-c', "import torch; assert torch.cuda.is_bf16_supported(), 'Full SFT baseline requires BF16-capable GPU'")
import shutil
MODEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for location in (Path('/kaggle/working'), MODEL_OUTPUT_ROOT):
    usage = shutil.disk_usage(location)
    print('Disk', location, 'free GiB:', round(usage.free / 1024**3, 2), flush=True)
for arm in ARMS:
    run('bash', SCRIPTS / 'train.sh', extra_env={
        'BASE_MODEL': str(BASE_MODEL), 'DATA_DIR': str(WORK / 'arms'),
        'OUTPUT_ROOT': str(MODEL_OUTPUT_ROOT), 'ARM': arm,
        'NUM_TRAIN_EPOCHS': str(NUM_TRAIN_EPOCHS), 'SEED': str(SEED),
        'LEARNING_RATE': str(LEARNING_RATE),
        'GRADIENT_ACCUMULATION_STEPS': str(GRADIENT_ACCUMULATION_STEPS)})


Inference một lượt, không dùng graph teacher; sampled fps được truyền cho
Qwen2.5. Lưu raw text, frame hash và token/time metadata. Không retry chọn output đẹp.

In [ ]:
predictions = []
model_checkpoints = {}
for arm in ARMS:
    checkpoints = list((MODEL_OUTPUT_ROOT / f'{arm}-seed{SEED}').rglob('config.json'))
    checkpoints = [p.parent for p in checkpoints
                   if p.parent.name.startswith('checkpoint-') and p.parent.name.split('-')[-1].isdigit()]
    assert checkpoints, f'No full checkpoint saved for {arm}'
    checkpoints = [max(checkpoints, key=lambda p: int(p.name.split('-')[-1]))]
    assert list(checkpoints[0].glob('*.safetensors')), 'Missing full checkpoint weights'
    assert not (checkpoints[0] / 'adapter_config.json').exists(), 'Expected full SFT checkpoint'
    model_checkpoints[arm] = checkpoints[0]
    output = WORK / f'{arm}-predictions.jsonl'
    run(sys.executable, SCRIPTS / 'run.py', 'predict', '--model', checkpoints[0],
        '--input', WORK / 'split/dev.jsonl', '--output', output)
    predictions.append(f'{arm}={output}')
run(sys.executable, SCRIPTS / 'evaluate.py', '--manifest', WORK / 'split/dev.jsonl',
    '--predictions', *predictions, '--reference', ARM, '--output', WORK / 'comparison.json')
print((WORK / 'comparison.json').read_text())
if MODE == 'full':
    source = model_checkpoints[ARM]
    destination = WORK / 'final_checkpoint'
    size = sum(p.stat().st_size for p in source.rglob('*') if p.is_file())
    free = shutil.disk_usage(WORK).free
    print('Final checkpoint GiB:', round(size / 1024**3, 2),
          'working free GiB:', round(free / 1024**3, 2), flush=True)
    assert free > size + 512 * 1024**2, 'Not enough persistent space for final checkpoint'
    shutil.copytree(source, destination)
    assert list(destination.glob('*.safetensors'))
    print('Persistent full checkpoint:', destination, flush=True)


Kết quả `smoke` chỉ kiểm tra kỹ thuật. Với `pilot`, xem joint F1@0.5 strict,
coverage. Một arm chỉ có điểm tuyệt đối, chưa có paired CI hoặc kết luận cải thiện.
Ghép predictions trên cùng dev manifest sau để so sánh. Đơn vị là fraction (0.02 = 2 điểm phần trăm).
`official_repaired` có policy sửa output của repo, cần báo riêng. CI bootstrap theo
video không bao phủ training-seed variance. Sau pilot: lặp 3 seeds, khóa cấu hình,
đánh giá official DAR test rồi mới cân nhắc GRPO lại. Đừng chọn metric thắng sau khi xem.

Nguồn: [STEP §3–4](https://arxiv.org/html/2412.00161v2),
[Video-of-Thought §3–5](https://arxiv.org/html/2501.03230v1).
Không có graph encoder, STEP iterative QRA self-training hoặc VoT verification loop
trong v0 này. Không có kết quả benchmark thật cho tới khi các cell GPU được chạy.